In [ ]:
from econml.orf import DMLOrthoForest, DROrthoForest
from econml.dml import CausalForestDML
from econml.sklearn_extensions.linear_model import WeightedLassoCVWrapper, WeightedLasso

# Helper imports
import numpy as np
from itertools import product
from sklearn.linear_model import Lasso, LogisticRegression
import matplotlib.pyplot as plt
from sklearn.linear_model import LassoCV
from scipy.special import expit 
import os
import pandas as pd
import urllib.request
from sklearn.preprocessing import StandardScaler

 
from sklearn.ensemble import (
    GradientBoostingRegressor,
    RandomForestRegressor
)

okabe_ito = {
    "orange":    "#E69F00",
    "sky_blue":  "#56B4E9",
    "green":     "#009E73",
    "yellow":    "#F0E442",
    "blue":      "#0072B2",
    "vermilion": "#D55E00",
    "pink":      "#CC79A7",
    "black":     "#000000",
}

oi_colors = list(okabe_ito.values())

%matplotlib inline

In [ ]:
# How do we simulate data?



In [4]:
# We are going to load the data from the EconML site:

oj_data = pd.read_csv("https://raw.githubusercontent.com/py-why/EconML/refs/heads/data/datasets/OrangeJuice/oj_large.csv")

oj_data.head()

,store,brand,week,logmove,feat,price,AGE60,EDUC,ETHNIC,INCOME,HHLARGE,WORKWOM,HVAL150,SSTRDIST,SSTRVOL,CPDIST5,CPWVOL5
0,2,tropicana,40,9.018695,0,3.87,0.232865,0.248935,0.11428,10.553205,0.103953,0.303585,0.463887,2.110122,1.142857,1.92728,0.376927
1,2,tropicana,46,8.723231,0,3.87,0.232865,0.248935,0.11428,10.553205,0.103953,0.303585,0.463887,2.110122,1.142857,1.92728,0.376927
2,2,tropicana,47,8.253228,0,3.87,0.232865,0.248935,0.11428,10.553205,0.103953,0.303585,0.463887,2.110122,1.142857,1.92728,0.376927
3,2,tropicana,48,8.987197,0,3.87,0.232865,0.248935,0.11428,10.553205,0.103953,0.303585,0.463887,2.110122,1.142857,1.92728,0.376927
4,2,tropicana,50,9.093357,0,3.87,0.232865,0.248935,0.11428,10.553205,0.103953,0.303585,0.463887,2.110122,1.142857,1.92728,0.376927


In [ ]:

# Prepare the data
Y = oj_data['logmove'].values  # The target is the log of the moving average of OJ sales
T = oj_data['feat'].values  # Featured is the treatment variable
scaler = StandardScaler()  # We will use the standard scaler

# We scale the continuous features (exclude feat since it's now the treatment)

W1 = scaler.fit_transform(oj_data[[c for c in oj_data.columns
                                   if c not in ['price', 'logmove', 'brand', 'week', 'store', 'feat']]].values)

# We one-hot the brand variables
W2 = pd.get_dummies(oj_data[['brand']]).values
W = np.concatenate([W1, W2], axis=1)

# This is the variable we are interested in exploring as a modifier

X = oj_data[['INCOME']].values

In [8]:
real_feat_rate = oj_data['feat'].mean()
real_feat_rate

np.float64(0.23726120150620097)

In [ ]:
# Before we do any fitting to the real data, we are going to see if our model could successfully find causal effects
# in fake data

# The idea is that we are going to use the real values of the confounders (The W) and the effect modifier (the X)
# While simulating the outcome variable


n, n_w = W.shape  
income_min, income_max = X.min(), X.max()

def true_cate(X):
    # Doesn't really matter what we put here, just something nonlinear that is generally
    # going to show a positive effect
    x_sc = (X - income_min) / (income_max - income_min)
    return 0.5 + 2.0 * np.sin(np.pi * x_sc)

TE_true = true_cate(X).ravel() # We calculate the true treatment effect



rng = np.random.default_rng(11111)

# Now we are going to pick 5 features from the confounder
# matrix W to be "real" confounders. These features each will
# have a random impact on both the treatment and outcome
# The hardest case is when the confounders impact both treatment and outcome
# so that is what we simulate

support_size = 5 # The number of features
support_idx = rng.choice(n_w, size=support_size, replace=False) # Select the indices

# First we will model the probability of receiving the treatment



gamma = rng.uniform(0.3, 0.8, size=support_size)
propensity_logit = W[:, support_idx] @ gamma
propensity = expit(propensity_logit - propensity_logit.mean() - 1.0)  # This will make it so about 25% is treated, which is close to the true rate
T_sim = np.random.binomial(1, propensity)

# --- Simulate outcome with known CATE ---
# Baseline outcome: same confounders (shared confounding)
beta = rng.uniform(0.5, 1.5, size=support_size)
mu_W = W[:, support_idx] @ beta

# Outcome = baseline + CATE * treatment + noise
epsilon = np.random.normal(0, 0.5, size=n)
Y_sim = mu_W + TE_true * T_sim + epsilon

# --- Diagnostics ---
print(f"Treatment rate: {T_sim.mean():.2f}")
print(f"Propensity range: [{propensity.min():.3f}, {propensity.max():.3f}]")
print(f"True ATE: {TE_true.mean():.3f}")
print(f"True CATE range: [{TE_true.min():.3f}, {TE_true.max():.3f}]")